In [2]:
"""
generate_synthetic_data.py
==========================
Generates a realistic synthetic dataset for site-level clinical trial
recruitment modeling.

Outputs (saved in the current folder):
    1. site_trial_history_extended.csv   (3,000 rows, master table)
    2. advertisement_strategy_catalog.csv (10 channels)
    3. protocol_catalog.csv               (3 protocols)

Run:
    python generate_synthetic_data.py

Author: Clinical Trial ML Team
"""

import numpy as np
import pandas as pd
from datetime import date, timedelta

# ======================================================================
# CONFIGURATION — change these if you want different sizes / seeds
# ======================================================================
N_ROWS = 3000
RANDOM_SEED = 42
OUTPUT_MASTER = "site_trial_history_extended.csv"
OUTPUT_AD_CATALOG = "advertisement_strategy_catalog.csv"
OUTPUT_PROTO_CATALOG = "protocol_catalog.csv"

np.random.seed(RANDOM_SEED)


# ======================================================================
# 1. PROTOCOL CATALOG
# ======================================================================
PROTOCOLS = {
    "PROTO-A_ONCO": dict(
        therapeutic_area="Oncology",
        phase="III",
        indication="NSCLC",
        sponsor_type="Pharma",
        protocol_complexity_score=8,
        num_visits=18,
        num_inclusion_criteria=12,
        num_exclusion_criteria=15,
        biomarker_required=True,
        prior_therapy_restriction=True,
        placebo_controlled=True,
    ),
    "PROTO-B_CARDIO": dict(
        therapeutic_area="Cardiology",
        phase="II",
        indication="Heart Failure",
        sponsor_type="Biotech",
        protocol_complexity_score=5,
        num_visits=9,
        num_inclusion_criteria=8,
        num_exclusion_criteria=10,
        biomarker_required=False,
        prior_therapy_restriction=True,
        placebo_controlled=True,
    ),
    "PROTO-C_NEURO": dict(
        therapeutic_area="Neurology",
        phase="II",
        indication="Alzheimer's",
        sponsor_type="Academic",
        protocol_complexity_score=7,
        num_visits=14,
        num_inclusion_criteria=10,
        num_exclusion_criteria=12,
        biomarker_required=True,
        prior_therapy_restriction=False,
        placebo_controlled=False,
    ),
}


# ======================================================================
# 2. ADVERTISEMENT CHANNEL CATALOG
# ======================================================================
AD_CHANNELS = {
    "adv_poster":                 dict(type="passive", cost=200,  reach="low",    effectiveness=0.15),
    "adv_pamphlet":               dict(type="passive", cost=150,  reach="low",    effectiveness=0.12),
    "adv_email":                  dict(type="active",  cost=500,  reach="high",   effectiveness=0.45),
    "adv_online":                 dict(type="active",  cost=1200, reach="high",   effectiveness=0.35),
    "adv_newsletter":             dict(type="passive", cost=300,  reach="medium", effectiveness=0.20),
    "adv_newspaper":              dict(type="passive", cost=800,  reach="medium", effectiveness=0.10),
    "adv_radio":                  dict(type="passive", cost=1500, reach="medium", effectiveness=0.08),
    "adv_tv":                     dict(type="passive", cost=5000, reach="high",   effectiveness=0.12),
    "adv_community_presentation": dict(type="active",  cost=400,  reach="medium", effectiveness=0.40),
    "adv_other":                  dict(type="active",  cost=600,  reach="varies", effectiveness=0.30),
}

CHANNEL_LABEL = {
    "adv_poster": "Poster",
    "adv_pamphlet": "Pamphlet",
    "adv_email": "Email",
    "adv_online": "Online",
    "adv_newsletter": "Newsletter",
    "adv_newspaper": "Newspaper",
    "adv_radio": "Radio",
    "adv_tv": "TV",
    "adv_community_presentation": "Community",
    "adv_other": "Other",
}


# ======================================================================
# 3. COUNTRY / REGULATORY MAP
# ======================================================================
COUNTRY_REG = {
    "USA":       dict(region="North America", authority="FDA",   ec_default="IRB", import_license=False),
    "Germany":   dict(region="Europe",        authority="EMA",   ec_default="CEC", import_license=True),
    "UK":        dict(region="Europe",        authority="MHRA",  ec_default="LEC", import_license=True),
    "Japan":     dict(region="Asia",          authority="PMDA",  ec_default="CEC", import_license=True),
    "Brazil":    dict(region="LATAM",         authority="Other", ec_default="LEC", import_license=True),
    "India":     dict(region="Asia",          authority="Other", ec_default="LEC", import_license=True),
    "Poland":    dict(region="Europe",        authority="EMA",   ec_default="LEC", import_license=True),
    "Spain":     dict(region="Europe",        authority="EMA",   ec_default="LEC", import_license=True),
    "Canada":    dict(region="North America", authority="Other", ec_default="IRB", import_license=False),
    "Australia": dict(region="APAC",          authority="Other", ec_default="LEC", import_license=True),
}
COUNTRIES = list(COUNTRY_REG.keys())
COUNTRY_PROBS = np.array([0.22, 0.14, 0.10, 0.10, 0.08, 0.08, 0.10, 0.08, 0.05, 0.05])
COUNTRY_PROBS = COUNTRY_PROBS / COUNTRY_PROBS.sum()


# ======================================================================
# 4. HELPER FUNCTIONS
# ======================================================================
def ts(d):
    """Convert any date-like object to a pandas Timestamp (safe for .quarter)."""
    return pd.Timestamp(d)


def sample_ec_timeline(ec_type):
    """Return (submission_to_approval_days, total_lead_days) for a given EC type."""
    if ec_type == "LEC":
        sub_delay = np.random.randint(10, 45)
        app_delay = np.random.randint(20, 70)
    elif ec_type == "CEC":
        sub_delay = np.random.randint(20, 90)
        app_delay = np.random.randint(45, 150)
    else:  # IRB
        sub_delay = np.random.randint(15, 60)
        app_delay = np.random.randint(30, 100)
    return sub_delay, app_delay


def sample_advertisement(protocol_complexity):
    """Return dict of 0/1 flags for each advertisement channel."""
    # Active channels more likely when protocol is 'medium complexity'
    active_bias = 0.15 + 0.05 * (5 - abs(protocol_complexity - 5))
    active_bias = float(np.clip(active_bias, 0.10, 0.45))
    passive_bias = 0.55

    probs = {
        "adv_poster": 0.75,
        "adv_pamphlet": 0.65,
        "adv_email": active_bias + 0.20,
        "adv_online": active_bias + 0.10,
        "adv_newsletter": passive_bias * 0.5,
        "adv_newspaper": passive_bias * 0.35,
        "adv_radio": passive_bias * 0.18,
        "adv_tv": passive_bias * 0.12,
        "adv_community_presentation": active_bias + 0.05,
        "adv_other": 0.15,
    }
    return {k: int(np.random.binomial(1, min(v, 0.95))) for k, v in probs.items()}


# ======================================================================
# 5. GENERATE THE MASTER DATASET
# ======================================================================
def generate_row(i):
    """Generate one synthetic trial record."""
    pid = np.random.choice(list(PROTOCOLS.keys()))
    p = PROTOCOLS[pid]

    # ---- Geography & regulatory ----
    country = np.random.choice(COUNTRIES, p=COUNTRY_PROBS)
    reg = COUNTRY_REG[country]
    region = reg["region"]
    authority = reg["authority"]
    ec_type = reg["ec_default"]
    if np.random.rand() < 0.25:  # 25% deviation from country default
        ec_type = "LEC" if ec_type == "CEC" else "CEC"
    import_license_required = reg["import_license"]

    # ---- Site / PI ----
    pi_experience = int(np.random.randint(2, 25))
    pi_prior_same_ind = int(np.random.poisson(3))
    coordinator_count = int(np.random.randint(1, 6))
    gcp_training_valid = int(np.random.binomial(1, 0.92))

    # ---- Patient pool ----
    base_pool = {"NSCLC": 420, "Heart Failure": 980, "Alzheimer's": 260}[p["indication"]]
    biomarker_penalty = 0.35 if p["biomarker_required"] else 1.0
    eligible_pool = int(max(np.random.normal(base_pool * biomarker_penalty, base_pool * 0.15), 20))
    competing = int(np.random.poisson(4))

    # ---- Timeline ----
    expected_duration_years = float(round(np.random.uniform(1.0, 4.0), 1))
    est_start = ts(date(2021, 1, 1) + timedelta(days=int(np.random.randint(0, 1000))))

    # ---- Ethics committee timeline ----
    ec_sub_delay, ec_app_delay = sample_ec_timeline(ec_type)
    ec_submission = est_start - pd.Timedelta(days=ec_sub_delay + ec_app_delay)
    ec_approval = ec_submission + pd.Timedelta(days=ec_app_delay)
    ec_approval_delay_days = ec_app_delay

    # ---- Regulatory timeline ----
    reg_sub_delay = np.random.randint(10, 60)
    reg_app_delay = np.random.randint(30, 180)
    reg_submission = ec_submission - pd.Timedelta(days=reg_sub_delay)
    reg_approval = reg_submission + pd.Timedelta(days=reg_app_delay)
    regulatory_delay_days = reg_app_delay

    import_delay = np.random.randint(30, 90) if import_license_required else 0
    num_ec_amendments = int(np.random.poisson(2))

    # ---- First patient in ----
    startup_delay = np.random.randint(5, 30)
    fpi_delay_extra = import_delay + num_ec_amendments * np.random.randint(3, 10)
    actual_fpi = max(ec_approval, reg_approval) + pd.Timedelta(
        days=startup_delay + fpi_delay_extra + int(np.random.normal(10, 15))
    )

    # ---- Recruitment window ----
    planned_window_days = int(expected_duration_years * 365)
    window_shrink = fpi_delay_extra + ec_approval_delay_days // 3
    recruitment_window_days = int(max(planned_window_days - window_shrink, 90))

    actual_lpi = actual_fpi + pd.Timedelta(days=recruitment_window_days)
    estimated_completion = actual_lpi + pd.Timedelta(days=int(np.random.normal(0, 30)))

    # ---- Derived timeline features ----
    fpi_delay_days = int((actual_fpi - est_start).days)
    schedule_slip_days = int((actual_lpi - estimated_completion).days)
    start_quarter = int(actual_fpi.quarter)   # safe on pd.Timestamp
    start_year = int(actual_fpi.year)

    # ---- Advertisement strategy ----
    adv = sample_advertisement(p["protocol_complexity_score"])
    adv_other_specify = "Patient registry outreach" if adv["adv_other"] else ""

    active_channels = ["adv_email", "adv_online", "adv_community_presentation", "adv_other"]
    passive_channels = ["adv_poster", "adv_pamphlet", "adv_newsletter",
                        "adv_newspaper", "adv_radio", "adv_tv"]

    num_active = int(sum(adv[c] for c in active_channels))
    num_passive = int(sum(adv[c] for c in passive_channels))
    active_outreach_flag = int(num_active > 0)
    passive_only_flag = int(num_active == 0 and num_passive > 0)
    digital_flag = int(adv["adv_email"] or adv["adv_online"])
    total_strategies = num_active + num_passive
    ad_cost_total = int(sum(AD_CHANNELS[c]["cost"] for c in AD_CHANNELS if adv[c]))

    # ---- Screen failure rate ----
    sf_base = 0.25 + 0.03 * p["protocol_complexity_score"]
    if p["biomarker_required"]:
        sf_base += 0.08
    if ec_type == "LEC":
        sf_base += 0.02
    screen_failure_rate = float(np.clip(np.random.normal(sf_base, 0.05), 0.05, 0.75))

    # ---- Advertisement effectiveness multiplier ----
    ad_effect = sum(AD_CHANNELS[c]["effectiveness"] * adv[c] for c in AD_CHANNELS if adv[c])
    ad_multiplier = (1 + 0.35 * np.log1p(ad_effect)) if ad_effect > 0 else 0.85

    # ---- Enrollment signal (target) ----
    enrollment_signal = (
        eligible_pool
        * (1 - screen_failure_rate)
        * (1 + 0.04 * pi_experience)
        * (1 + 0.06 * pi_prior_same_ind)
        * (0.85 ** competing)
        * (0.90 ** (p["protocol_complexity_score"] - 5))
        * ad_multiplier
        * (recruitment_window_days / 365) ** 0.6
        * np.random.normal(1.0, 0.15)
    )
    actual_enrolled = int(np.clip(enrollment_signal, 0, 400))
    enrollment_rate_per_month = round(actual_enrolled / max(recruitment_window_days / 30, 1), 2)

    # ---- Best channel label (for strategy-selection model) ----
    channel_eff = {c: AD_CHANNELS[c]["effectiveness"] * adv[c] for c in AD_CHANNELS}
    proto_bias = {
        "PROTO-A_ONCO":  {"adv_community_presentation": 1.3, "adv_email": 1.2, "adv_online": 0.9},
        "PROTO-B_CARDIO": {"adv_online": 1.3, "adv_email": 1.1, "adv_community_presentation": 1.0},
        "PROTO-C_NEURO": {"adv_community_presentation": 1.4, "adv_newsletter": 1.3, "adv_email": 1.0},
    }[pid]
    weighted = {c: channel_eff[c] * proto_bias.get(c, 1.0) for c in channel_eff}
    best_channel = max(weighted, key=weighted.get) if any(weighted.values()) else "none"

    # ---- Build the row ----
    row = {
        # Identifiers
        "trial_id": f"SITE-{i:05d}",
        "protocol_id": pid,
        "country": country,
        "region": region,

        # Protocol attributes
        "therapeutic_area": p["therapeutic_area"],
        "phase": p["phase"],
        "indication": p["indication"],
        "sponsor_type": p["sponsor_type"],
        "protocol_complexity_score": p["protocol_complexity_score"],
        "num_visits": p["num_visits"],
        "num_inclusion_criteria": p["num_inclusion_criteria"],
        "num_exclusion_criteria": p["num_exclusion_criteria"],
        "biomarker_required": int(p["biomarker_required"]),
        "prior_therapy_restriction": int(p["prior_therapy_restriction"]),
        "placebo_controlled": int(p["placebo_controlled"]),

        # Site / PI
        "site_pi_id": f"PI-{np.random.randint(1, 30):03d}",
        "pi_experience_years": pi_experience,
        "pi_prior_trials_same_indication": pi_prior_same_ind,
        "coordinator_count": coordinator_count,
        "gcp_training_valid": gcp_training_valid,

        # Patient pool / competition
        "competing_trials_same_indication_region": competing,
        "eligible_patient_pool_estimate": eligible_pool,

        # Timeline raw
        "expected_duration_years": expected_duration_years,
        "estimated_start_date": est_start,
        "actual_first_patient_in": actual_fpi,
        "actual_last_patient_in": actual_lpi,
        "estimated_completion_date": estimated_completion,

        # Timeline derived
        "fpi_delay_days": fpi_delay_days,
        "recruitment_window_days": recruitment_window_days,
        "schedule_slip_days": schedule_slip_days,
        "start_quarter": start_quarter,
        "start_year": start_year,

        # Regulatory
        "regulatory_authority_type": authority,
        "regulatory_submission_date": reg_submission,
        "regulatory_approval_date": reg_approval,
        "regulatory_delay_days": regulatory_delay_days,
        "import_license_required": int(import_license_required),
        "ec_type": ec_type,
        "ec_submission_date": ec_submission,
        "ec_approval_date": ec_approval,
        "ec_approval_delay_days": ec_approval_delay_days,
        "ec_submission_to_fpi_days": int((actual_fpi - ec_submission).days),
        "num_ec_amendments": num_ec_amendments,

        # Advertisement raw
        **adv,
        "adv_other_specify": adv_other_specify,

        # Advertisement derived
        "num_active_strategies": num_active,
        "num_passive_strategies": num_passive,
        "total_strategies": total_strategies,
        "active_outreach_flag": active_outreach_flag,
        "passive_only_flag": passive_only_flag,
        "digital_flag": digital_flag,
        "ad_cost_total": ad_cost_total,

        # Outcome / target
        "screen_failure_rate": round(screen_failure_rate, 3),
        "actual_enrolled": actual_enrolled,
        "enrollment_rate_per_month": enrollment_rate_per_month,

        # Strategy-selection label
        "best_channel": best_channel,
        "best_channel_label": CHANNEL_LABEL.get(best_channel, "None"),
    }
    return row


# ======================================================================
# 6. RUN GENERATION
# ======================================================================
def main():
    print("=" * 70)
    print("Synthetic Clinical Trial Recruitment Dataset Generator")
    print("=" * 70)
    print(f"Rows to generate : {N_ROWS:,}")
    print(f"Protocols        : {len(PROTOCOLS)}")
    print(f"Countries        : {len(COUNTRIES)}")
    print(f"Ad channels      : {len(AD_CHANNELS)}")
    print("-" * 70)

    print("Generating rows ...", end=" ", flush=True)
    rows = [generate_row(i) for i in range(N_ROWS)]
    df = pd.DataFrame(rows)
    print("done.")

    # Save master table
    df.to_csv(OUTPUT_MASTER, index=False)
    print(f"✅ Saved master table         : {OUTPUT_MASTER}  ({df.shape[0]} rows × {df.shape[1]} cols)")

    # Save ad catalog
    ad_df = pd.DataFrame(AD_CHANNELS).T.reset_index().rename(columns={"index": "channel"})
    ad_df["channel_label"] = ad_df["channel"].map(CHANNEL_LABEL)
    ad_df.to_csv(OUTPUT_AD_CATALOG, index=False)
    print(f"✅ Saved ad strategy catalog  : {OUTPUT_AD_CATALOG}  ({ad_df.shape[0]} channels)")

    # Save protocol catalog
    proto_df = pd.DataFrame(PROTOCOLS).T.reset_index().rename(columns={"index": "protocol_id"})
    proto_df.to_csv(OUTPUT_PROTO_CATALOG, index=False)
    print(f"✅ Saved protocol catalog     : {OUTPUT_PROTO_CATALOG}  ({proto_df.shape[0]} protocols)")

    # ---- Summary stats ----
    print("-" * 70)
    print("Quick validation:")
    print(f"  • Date range             : {df['actual_first_patient_in'].min().date()} → "
          f"{df['actual_last_patient_in'].max().date()}")
    print(f"  • Mean enrollment        : {df['actual_enrolled'].mean():.1f} subjects "
          f"(range {df['actual_enrolled'].min()}–{df['actual_enrolled'].max()})")
    print(f"  • Mean FPI delay         : {df['fpi_delay_days'].mean():.0f} days")
    print(f"  • Mean recruitment window: {df['recruitment_window_days'].mean():.0f} days")
    print(f"  • Active outreach share  : {df['active_outreach_flag'].mean():.1%}")
    print(f"  • EC type distribution   : "
          + ", ".join(f"{k}={v}" for k, v in df['ec_type'].value_counts().items()))

    print("-" * 70)
    print("Enrollment by protocol:")
    proto_summary = df.groupby("protocol_id")["actual_enrolled"].agg(
        ["count", "mean", "median", "min", "max"]
    ).round(1)
    print(proto_summary.to_string())

    print("-" * 70)
    print("Top recommended channels (best_channel_label):")
    print(df["best_channel_label"].value_counts().head(5).to_string())

    print("=" * 70)
    print("🎉 Done. You can now load the CSV files into your model.")
    print("=" * 70)


if __name__ == "__main__":
    main()

Synthetic Clinical Trial Recruitment Dataset Generator
Rows to generate : 3,000
Protocols        : 3
Countries        : 10
Ad channels      : 10
----------------------------------------------------------------------
Generating rows ... done.
✅ Saved master table         : site_trial_history_extended.csv  (3000 rows × 66 cols)
✅ Saved ad strategy catalog  : advertisement_strategy_catalog.csv  (10 channels)
✅ Saved protocol catalog     : protocol_catalog.csv  (3 protocols)
----------------------------------------------------------------------
Quick validation:
  • Date range             : 2020-12-03 → 2027-12-02
  • Mean enrollment        : 182.0 subjects (range 3–400)
  • Mean FPI delay         : 66 days
  • Mean recruitment window: 849 days
  • Active outreach share  : 84.0%
  • EC type distribution   : LEC=1272, CEC=1118, IRB=610
----------------------------------------------------------------------
Enrollment by protocol:
                count   mean  median  min  max
protocol_id    